In [12]:
import chess.pgn
import io
import json
import os
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

folders = [f.path for f in os.scandir(".") if f.is_dir()]
folders.sort()

def flat_json(game: dict):
    uuid: str = game.get("uuid", "")
    pgn: str = game.get("pgn", "")
    time_control: str = game.get("time_control", "")
    time_class: str = game.get("time_class", "")
    rated: bool = game.get("rated", False)
    initial_setup: str = game.get("initial_setup", "")
    fen: str = game.get("fen", "")
    tcn: str = game.get("tcn", "")
    rules: str = game.get("rules", "")
    end_time: int = game.get("end_time", "")
    white_accuracy: float = game.get("accuracies", {}).get("white", 0)
    white_rating: int = game.get("white", {}).get("rating", 0)
    white_result: str = game.get("white", {}).get("result", "")
    white_username: str = game.get("white", {}).get("username", "").lower()
    black_accuracy: float = game.get("accuracies", {}).get("black", 0)
    black_rating: int = game.get("black", {}).get("rating", 0)
    black_result: str = game.get("black", {}).get("result", "")
    black_username: str = game.get("black", {}).get("username", "").lower()
    pgn_info = chess.pgn.read_game(io.StringIO(pgn))
    pgn_headers = pgn_info.headers if pgn_info is not None else chess.pgn.Headers()
    opening: str = pgn_headers.get("ECO", "")
    opening_url: str = pgn_headers.get("ECOUrl", "")
    opening_name: str = " ".join(opening_url.split("/")[-1].split("-"))
    return {
        "uuid": uuid,
        "end_time": end_time,
        "time_control": time_control,
        "time_class": time_class,
        "rated": rated,
        "initial_setup": initial_setup,
        "fen": fen,
        "tcn": tcn,
        "rules": rules,
        "white_accuracy": white_accuracy,
        "white_rating": white_rating,
        "white_result": white_result,
        "white_username": white_username,
        "black_accuracy": black_accuracy,
        "black_rating": black_rating,
        "black_result": black_result,
        "black_username": black_username,
        "pgn": pgn,
        "opening": opening,
        "opening_name": opening_name,
    }


for folder in tqdm(folders):
    # Parse Games in JSON
    games_file = open(f"{folder}/json/all.json", "r")
    games_json: list[dict] = json.load(games_file)
    games_file.close()
    if len(games_json) >= 10000:
        continue
    games_list: list[dict] = list(map(lambda game: flat_json(game), games_json))
    Path(f"{folder}/csv").mkdir(parents=True, exist_ok=True)
    # Games with PGN
    all_df = pd.DataFrame(games_list)
    all_df.to_csv(f"{folder}/csv/all.csv", index=False)
    # Games without PGN
    games_df = all_df.drop("pgn", axis=1)
    games_df.to_csv(f"{folder}/csv/games.csv", index=False)
    # Check output
    csv_size = os.path.getsize(f"{folder}/csv/games.csv")
    json_size = os.path.getsize(f"{folder}/json/all.json")
    print(
        folder,
        "{:,}".format(len(games_json)),
        f"csv: {round(csv_size/(pow(1024,2)), 2)} MB",
        f"json: {round(json_size/(pow(1024,2)), 2)} MB",
    )

  0%|          | 0/31 [00:00<?, ?it/s]

./anishgiri 1,429 csv: 0.65 MB json: 6.03 MB
./azerichess 3,760 csv: 1.62 MB json: 14.76 MB
./chefshouse 595 csv: 0.25 MB json: 2.08 MB
./chesswarrior7197 6,154 csv: 2.78 MB json: 26.08 MB
./denlaz 6,177 csv: 2.73 MB json: 26.18 MB
./dominguezonyoutube 4,349 csv: 1.95 MB json: 17.95 MB
./fabianocaruana 4,605 csv: 2.07 MB json: 19.71 MB
./gmwso 9,557 csv: 3.71 MB json: 27.72 MB
./grischuk 4,458 csv: 2.0 MB json: 19.01 MB
./gukeshdommaraju 5,642 csv: 2.42 MB json: 21.88 MB
./lachesisq 7,177 csv: 3.22 MB json: 29.68 MB
./levonaronian 1,058 csv: 0.48 MB json: 4.61 MB
./liemle 2,851 csv: 1.25 MB json: 11.75 MB
./lyonbeast 5,372 csv: 2.35 MB json: 21.67 MB
./magnuscarlsen 5,217 csv: 2.33 MB json: 22.1 MB
./rpragchess 8,604 csv: 3.66 MB json: 33.88 MB
./sergeykarjakin 489 csv: 0.23 MB json: 2.07 MB
./thevish 66 csv: 0.03 MB json: 0.27 MB
./tradjabov 1,343 csv: 0.59 MB json: 5.5 MB
./veselintopalov359 25 csv: 0.01 MB json: 0.1 MB
./viditchess 3,631 csv: 1.62 MB json: 14.95 MB
./vladimirkramnik